# Latih NLLB-200 manga seperti DeepL (Google Colab)

Notebook ini menjalankan alur lengkap dari folder `training/`:

1. mengumpulkan pasangan dialog Jepang → Indonesia,
2. LoRA fine-tune `facebook/nllb-200-distilled-600M`,
3. ekspor ke CTranslate2 int8,
4. evaluasi dan unduh model siap pakai.

**Yang dibutuhkan**

- Akun Google dan runtime GPU (T4 gratis sudah cukup).
- Project MangaFlow dalam ZIP, termasuk folder `server/`, `training/`, dan `data/jobs/` bila ada.
- Kunci DeepL **opsional**. Tanpa kunci, pilih guru gratis `google` atau `mymemory`.

> Model belum terlatih sampai semua sel dijalankan. Perkiraan waktu di T4: 20–60 menit untuk 1.000–3.000 pasangan.

## 1. Cek GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip -q install "transformers>=4.46,<5" "datasets>=2.19" "peft>=0.12" "accelerate>=0.33" sentencepiece ctranslate2 sacrebleu

## 2. Unggah project MangaFlow

Buat ZIP dari folder `manga-translator` (boleh tanpa `data/models/` agar lebih kecil), lalu unggah di sel berikut.

In [ ]:
from google.colab import files
print("Pilih ZIP project, misalnya manga-translator.zip")
uploaded = files.upload()

In [ ]:
import os, zipfile
from pathlib import Path

for name in uploaded:
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall("/content")

roots = sorted({
    p.parent for p in Path("/content").rglob("app.py")
    if (p.parent / "server" / "translate.py").exists()
})
if not roots:
    raise SystemExit("app.py tidak ditemukan. Pastikan ZIP berisi folder manga-translator.")
os.chdir(roots[0])
print("Project:", os.getcwd())

## 3. Kumpulkan pasangan latih

Skrip membaca teks Jepang dari `data/jobs/*/pages/*/regions.json`, lalu menerjemahkannya dengan guru pilihan.

- `deepl` — paling dekat dengan gaya yang ingin ditiru, butuh API key.
- `google` — gratis, kualitas lumayan, rawan rate limit.
- `mymemory` — gratis, kuota harian terbatas, tidak mendukung deteksi bahasa otomatis.

**Target data:** minimal 500 pasangan; 3.000+ lebih stabil. Satu halaman manga biasanya menghasilkan 3–8 baris dialog.

In [ ]:
TEACHER = "deepl"          # "deepl", "google", atau "mymemory"
MAX_LINES = 3000
BUDGET_CHARS = 400000
DELAY = 0.0 if TEACHER == "deepl" else 0.3

if TEACHER == "deepl":
    import os, getpass
    if not os.environ.get("DEEPL_API_KEY"):
        os.environ["DEEPL_API_KEY"] = getpass.getpass("DeepL API key: ")

In [ ]:
!python training/collect_deepl.py --teacher {TEACHER} --max-lines {MAX_LINES} --budget-chars {BUDGET_CHARS} --delay {DELAY} --dry-run

In [ ]:
!python training/collect_deepl.py --teacher {TEACHER} --max-lines {MAX_LINES} --budget-chars {BUDGET_CHARS} --delay {DELAY} --append

### Opsional: gabungkan pasangan manual/koreksi

Unggah `pairs.jsonl` berisi pasangan yang sudah kamu periksa, lalu gabungkan tanpa menerjemahkan ulang.

In [ ]:
from google.colab import files
manual = files.upload()
if manual:
    import shutil
    name = next(iter(manual))
    shutil.copy(name, "training/data/manual_pairs.jsonl")
    print("Disalin ke training/data/manual_pairs.jsonl")

In [ ]:
!python training/collect_deepl.py --seed training/data/manual_pairs.jsonl --seed-only --append

In [ ]:
import json
from pathlib import Path
p = Path("training/data/pairs.jsonl")
rows = [json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]
print("Total pasangan:", len(rows))
if len(rows) < 500:
    print("PERINGATAN: dataset masih kecil. Tambahkan halaman manga atau pasangan manual.")
for row in rows[:5]:
    print(" ", row["src"], "->", row["tgt"])

### Periksa kualitas data

Hapus atau koreksi baris OCR sampah sebelum melatih.

In [ ]:
!python training/check_dataset.py --data training/data/pairs.jsonl --show 12

## 4. Fine-tune LoRA

Default: 4 epoch, batch 4, gradient accumulation 4, fp16. Di T4 ini biasanya muat.

Jika muncul **CUDA out of memory**, ubah menjadi `--batch-size 2 --grad-accum 8`, atau turunkan `--max-length 128`.

In [ ]:
!python training/finetune_nllb.py \
  --data training/data/pairs.jsonl \
  --output training/out/nllb-manga-lora \
  --epochs 4 \
  --batch-size 4 \
  --grad-accum 4 \
  --lr 1e-4 \
  --fp16 \
  --val-out training/data/test.jsonl

## 5. Ekspor ke CTranslate2 int8

Langkah ini menggabungkan adapter LoRA ke model penuh, lalu mengonversinya ke format yang dipakai MangaFlow.

In [ ]:
!python training/export_ct2.py --adapter training/out/nllb-manga-lora --force

## 6. Evaluasi

Bandingkan model dasar dan model latihan pada data uji yang tidak dipakai saat training.

In [ ]:
!python training/evaluate.py \
  --data training/data/test.jsonl \
  --model data/models/nllb-200-distilled-600M-ct2-int8 \
  --compare data/models/nllb-200-manga-ct2-int8 \
  --limit 150

## 7. Unduh model

Unduh ZIP, lalu ekstrak ke `manga-translator/data/models/` di komputer lokal.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/nllb-manga-ct2-int8",
    "zip",
    root_dir="data/models",
    base_dir="nllb-200-manga-ct2-int8",
)
files.download("/content/nllb-manga-ct2-int8.zip")

## 8. Pakai di MangaFlow

1. Ekstrak ZIP sehingga struktur menjadi `data/models/nllb-200-manga-ct2-int8/model.bin`.
2. Jalankan ulang server: `python app.py`.
3. Di UI, pilih mesin **NLLB-200 manga (latihan)** atau biarkan fallback CJK memilihnya otomatis.
4. Pilih **Gaya bahasa → Santai** untuk dialog manga.

Jika model ada di folder lain, set:

```bash
set MANGA_NLLB_TUNED_DIR=D:\model\nllb-manga-ct2-int8
python app.py
```

## Troubleshooting

**`Dependensi latihan belum terpasang`** — jalankan ulang sel `%pip install`.

**CUDA out of memory** — `--batch-size 2 --grad-accum 8`, atau `--max-length 128`.

**Hasil menghafal / aneh** — dataset terlalu kecil atau kotor. Periksa `pairs.jsonl`, buang baris OCR sampah, dan tambah data.

**Model tidak muncul di UI** — pastikan `model.bin`, `sentencepiece.bpe.model`, `shared_vocabulary.txt`, dan `config.json` ada di folder model.